# 01 - LQR

This classroom notebook starts with open-loop motion, adds feedback by hand, and then derives the discrete LQR feedback law. The last experiment clips the LQR input and shows why local saturation is not the same thing as solving a constrained optimal-control problem.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import solve_discrete_are

np.set_printoptions(precision=3, suppress=True)

## Simple System Setup

We begin with the scalar integrator x[k+1] = x[k] + u[k]. With no feedback, an input sequence is just played open loop.

In [ ]:
A = 1.0
B = 1.0
steps = 12
x = np.zeros(steps + 1)
u = np.zeros(steps)
x[0] = 3.0

for k in range(steps):
    u[k] = -0.2
    x[k + 1] = A * x[k] + B * u[k]

plt.figure(figsize=(6, 3))
plt.step(range(steps), u, where="post", label="u")
plt.plot(range(steps + 1), x, "o-", label="x")
plt.axhline(0.0, color="k", linewidth=0.8)
plt.grid(True, alpha=0.25)
plt.legend()
plt.title("Open-loop scalar behavior")
plt.show()

## Manual Feedback Idea

A simple feedback law chooses u[k] from the current state. Here u = -Kx. Larger K reacts faster, but it also asks for larger inputs.

In [ ]:
plt.figure(figsize=(6, 3))
for K_manual in [0.2, 0.5, 0.9]:
    x = np.zeros(steps + 1)
    u = np.zeros(steps)
    x[0] = 3.0
    for k in range(steps):
        u[k] = -K_manual * x[k]
        x[k + 1] = A * x[k] + B * u[k]
    plt.plot(range(steps + 1), x, "o-", label=f"K = {K_manual}")
plt.axhline(0.0, color="k", linewidth=0.8)
plt.grid(True, alpha=0.25)
plt.legend()
plt.title("Manual scalar feedback")
plt.show()

## Scalar LQR

LQR chooses K from a cost. For the scalar teaching case we use A = 1, B = 1, Q = 1, R = 1.

In [ ]:
A = np.array([[1.0]])
B = np.array([[1.0]])
Q = np.array([[1.0]])
R = np.array([[1.0]])

P = solve_discrete_are(A, B, Q, R)
K = np.linalg.solve(R + B.T @ P @ B, B.T @ P @ A)

print("P from DARE =", P)
print("LQR gain K =", K)
print("closed-loop A - B K =", A - B @ K)

## Visible Scalar Closed-Loop Simulation

Now the simulation loop uses the LQR gain directly.

In [ ]:
steps = 12
x = np.zeros(steps + 1)
u = np.zeros(steps)
x[0] = 3.0

for k in range(steps):
    u[k] = -(K @ np.array([x[k]]))[0]
    x[k + 1] = (A @ np.array([x[k]]) + B @ np.array([u[k]]))[0]

plt.figure(figsize=(6, 3))
plt.plot(range(steps + 1), x, "o-", label="x")
plt.step(range(steps), u, where="post", label="u")
plt.axhline(0.0, color="k", linewidth=0.8)
plt.grid(True, alpha=0.25)
plt.legend()
plt.title("Scalar LQR")
plt.show()

## Double Integrator LQR

The next example is the discrete double integrator with dt = 1.

In [ ]:
A = np.array([[1.0, 1.0],
              [0.0, 1.0]])
B = np.array([[0.0],
              [1.0]])
Q = np.eye(2)
R = np.array([[1.0]])

P = solve_discrete_are(A, B, Q, R)
K = np.linalg.solve(R + B.T @ P @ B, B.T @ P @ A)

print("P =\n", P)
print("K =", K)
print("closed-loop eigenvalues =", np.linalg.eigvals(A - B @ K))

## Visible 2D LQR Behavior

The state is position and velocity. The controller accelerates first, then brakes so both states approach zero.

In [ ]:
steps = 20
X = np.zeros((steps + 1, 2))
U = np.zeros(steps)
X[0] = np.array([5.0, 0.0])

for k in range(steps):
    U[k] = -(K @ X[k])[0]
    X[k + 1] = A @ X[k] + B[:, 0] * U[k]

t = np.arange(steps + 1)
fig, axes = plt.subplots(3, 1, figsize=(7, 6), sharex=True)
axes[0].plot(t, X[:, 0], "o-")
axes[0].set_ylabel("position")
axes[1].plot(t, X[:, 1], "o-")
axes[1].set_ylabel("velocity")
axes[2].step(t[:-1], U, where="post")
axes[2].set_ylabel("input")
axes[2].set_xlabel("step")
for ax in axes:
    ax.grid(True, alpha=0.25)
plt.show()

## Saturated LQR

Real actuators have limits. A common first attempt is to compute the LQR input and clip it.

In [ ]:
u_min = -0.8
u_max = 0.8
steps = 24
X_sat = np.zeros((steps + 1, 2))
U_sat = np.zeros(steps)
X_unsat = np.zeros((steps + 1, 2))
U_unsat = np.zeros(steps)
X_sat[0] = np.array([8.0, 0.0])
X_unsat[0] = X_sat[0].copy()

for k in range(steps):
    U_unsat[k] = -(K @ X_unsat[k])[0]
    X_unsat[k + 1] = A @ X_unsat[k] + B[:, 0] * U_unsat[k]

    raw = -(K @ X_sat[k])[0]
    U_sat[k] = np.clip(raw, u_min, u_max)
    X_sat[k + 1] = A @ X_sat[k] + B[:, 0] * U_sat[k]

fig, axes = plt.subplots(2, 1, figsize=(7, 5), sharex=True)
axes[0].plot(range(steps + 1), X_unsat[:, 0], label="LQR position")
axes[0].plot(range(steps + 1), X_sat[:, 0], label="saturated LQR position")
axes[1].step(range(steps), U_unsat, where="post", label="LQR input")
axes[1].step(range(steps), U_sat, where="post", label="saturated input")
axes[1].axhline(u_min, color="k", linestyle="--", linewidth=0.9)
axes[1].axhline(u_max, color="k", linestyle="--", linewidth=0.9)
axes[0].grid(True, alpha=0.25)
axes[1].grid(True, alpha=0.25)
axes[0].legend()
axes[1].legend()
axes[1].set_xlabel("step")
plt.show()

## Final Comparison

The clipped input respects the local actuator limit, but it does not choose an input sequence by predicting future constraints.

In [ ]:
x_limit = 3.0
steps = 24
X_sat = np.zeros((steps + 1, 2))
U_sat = np.zeros(steps)
X_sat[0] = np.array([2.5, 1.0])

for k in range(steps):
    raw = -(K @ X_sat[k])[0]
    U_sat[k] = np.clip(raw, u_min, u_max)
    X_sat[k + 1] = A @ X_sat[k] + B[:, 0] * U_sat[k]

fig, axes = plt.subplots(2, 1, figsize=(7, 5), sharex=True)
axes[0].plot(range(steps + 1), X_sat[:, 0], "o-", label="position")
axes[0].axhline(x_limit, color="tab:red", linestyle="--", label="example position limit")
axes[0].axhline(-x_limit, color="tab:red", linestyle="--")
axes[0].set_ylabel("position")
axes[1].step(range(steps), U_sat, where="post", label="clipped LQR input")
axes[1].axhline(u_min, color="k", linestyle="--", linewidth=0.9)
axes[1].axhline(u_max, color="k", linestyle="--", linewidth=0.9)
axes[1].set_ylabel("input")
axes[1].set_xlabel("step")
for ax in axes:
    ax.grid(True, alpha=0.25)
    ax.legend(loc="best")
plt.show()

print("maximum input magnitude:", np.max(np.abs(U_sat)))
print("maximum position magnitude:", np.max(np.abs(X_sat[:, 0])))

Saturated LQR enforces the input bound locally, but it does not solve the constrained optimal-control problem. This motivates MPC.